In [1]:
# Cài đặt thư viện cần thiết
!pip install bertopic sentence-transformers umap-learn hdbscan pandas numpy -q

import sys
import pandas as pd
import numpy as np

# Thêm đường dẫn chứa file bertopic_model.py vào sys.path để import được
# !!! THAY ĐỔI 'your-dataset-name' thành tên dataset thực tế của bạn trên Kaggle
sys.path.append('/kaggle/input/datasets/hoangquancs04222/data-proccessed-score') 
from bertopic_model import VietnameseBERTopicModel

# ==========================================
# 1. LOAD & CHUẨN BỊ DATA
# ==========================================
print("Đang load dữ liệu...")
# !!! THAY ĐỔI tên file csv cho đúng
data_path = '/kaggle/input/datasets/hoangquancs04222/data-proccessed-score/stg_posts_core.csv' 
df = pd.read_csv(data_path)

# Lọc bỏ các dòng không có segmented_text hoặc post_id
df = df.dropna(subset=['segmented_text', 'post_id'])

# Chuyển đổi thành List như class yêu cầu
documents = df['segmented_text'].tolist()
post_ids = df['post_id'].tolist()
print(f"Tổng số bài viết đưa vào train: {len(documents)}")

# ==========================================
# 2. KHỞI TẠO MODEL VỚI THAM SỐ "CHÂN ÁI"
# ==========================================
model = VietnameseBERTopicModel(
    embedding_model="vinai/phobert-base",
    n_neighbors=30,
    n_components=10,
    min_dist=0.0,
    min_cluster_size=8,
    min_samples=1,
    nr_topics="auto", # Bước 1: Để model tự do quét toàn bộ cấu trúc (~145 topics)
    use_gpu=True,
    verbose=True
)

# ==========================================
# 3. TIẾN HÀNH TRAINING (Tự động Encode + UMAP + HDBSCAN)
# ==========================================
print("\nBắt đầu quá trình Training...")
# Quá trình này sẽ tận dụng GPU T4 trên Kaggle để encode PhoBERT rất nhanh
topics, probs = model.fit(documents)

# ==========================================
# 4. 🪄 BƯỚC "MA THUẬT": GỘP TOPICS CHO DASHBOARD
# ==========================================
print("\nBắt đầu thực hiện phép màu: Gộp topics về 70...")
model.reduce_topics(documents, nr_topics=70)

# Tính điểm Coherence (tuỳ chọn, để report cho Sếp nếu cần)
coherence_score = model.calculate_coherence(documents)
print(f"\nĐiểm Coherence cuối cùng: {coherence_score:.4f}")

# ==========================================
# 5. XUẤT KẾT QUẢ VÀ LƯU MODEL VÀO /WORKING/
# ==========================================
# Thư mục /kaggle/working/ là nơi bạn có thể download file sau khi chạy xong
output_dir = "/kaggle/working/dashboard_outputs"

print("\nĐang xuất file CSV phục vụ Database/Dashboard...")
# Bảng 1: Map từng bài viết với Topic ID
df_post_topics = model.export_post_topics(
    post_ids=post_ids, 
    output_path=f"{output_dir}/stg_post_topics.csv",
    coherence_score=coherence_score
)

# Bảng 2: Từ điển Topics (ID, Labels, Keywords)
df_topics = model.export_stg_topics(
    output_path=f"{output_dir}/stg_topics.csv",
    coherence_score=coherence_score,
    model_version="bertopic_phobert_v_dashboard"
)

# Lưu lại toàn bộ cục Model dạng file Pickle để sau này inference (predict data mới)
model.save(f"{output_dir}/final_bertopic_model")

print("\n🎉 XONG! TẤT CẢ FILE ĐÃ SẴN SÀNG Ở MỤC OUTPUT CỦA KAGGLE (bên phải màn hình)!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.4 MB/s eta 0:00:0000:01


2026-05-02 15:57:00.433971: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777737420.903367      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777737421.004504      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777737422.068574      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777737422.068617      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777737422.068620      57 computation_placer.cc:177] computation placer alr

Đang load dữ liệu...


Tổng số bài viết đưa vào train: 164366
🚀 GPU detected: Tesla T4

[1/4] Loading embedding model: vinai/phobert-base


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

...

2026-05-02 15:59:11,965 - BERTopic - Embedding - Transforming documents to embeddings.


✅ Embedding model loaded (device: cuda)

[2/4] Configuring UMAP
  - n_neighbors: 30
  - n_components: 10
  - min_dist: 0.0
✅ UMAP configured

[3/4] Configuring HDBSCAN
  - min_cluster_size: 8
  - min_samples: 1
✅ HDBSCAN configured

[4/4] Building BERTopic pipeline
✅ BERTopic pipeline ready

Model initialized successfully!


Bắt đầu quá trình Training...

TRAINING BERTOPIC MODEL
Documents: 164366
Device: cuda

Starting training...
  [1/4] Encoding documents (PhoBERT)...


Batches:   0%|          | 0/5137 [00:00<?, ?it/s]

/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [4,0,0], thread: [32,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [4,0,0], thread: [33,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [0,0,0], thread: [0,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [0,0,0], thread: [1,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [5,0,0], thread: [38,0,0] Assertion `idx_dim >= 0 && idx_dim < index_siz

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
